# KBEL demo: `extract`, `link` and `construct`

This notebook shows the three main methods of `KBEL`:

| Method | Input | Output |
|--------|-------|--------|
| `link` | a `Mention` | the knowledge-base entities it refers to |
| `extract` | a text | `Triple(subject, predicate, object)` strings |
| `construct` | a document | KIF `Statement`s whose subject, property and value were all linked |

## Setup

In [1]:
import os
os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
os.environ.setdefault('TRANSFORMERS_VERBOSITY', 'error')

import logging
logging.basicConfig(level=logging.ERROR)

try:                      
    import dotenv
    dotenv.load_dotenv()
except ImportError:
    pass

from kbel.core.kbel import KBEL
from kbel.core.mention import Mention, EntityType

# Plugins register themselves on import.
from kbel.disambiguators import naive, similarity, llm 
from kbel.extractors import mrebel, openie

In [2]:
def entity(result):
    """`label (id)` of a LinkResult."""
    return f"{result.candidate.label} ({result.candidate.iri.rsplit('/', 1)[-1]})"

## 1. `link`: from a mention to an entity

A `Mention` has the `term` to link, the sentence it appears in (`text`) and
whether it is an item or a property. `link` searches the knowledge base for
candidates and lets the **disambiguator** choose.

In [21]:
kbel = KBEL('wikidata', disambiguator='naive')

mention = Mention(term='Python', text='Python is a programming language used for coding.')

for result in kbel.link(mention):
    print(entity(result), '-', result.candidate.description)

Python (Q28865) - general-purpose programming language


The `naive` disambiguator ignores the sentence: it always picks the candidate
whose label matches the term. The `sim` disambiguator compares the sentence
with each candidate's description, so the same term can link to different
entities:

In [34]:
snake = Mention(term='Python', text='The python is a large snake that kills its prey by constriction.')

for name in ('naive', 'sim'):
    kbel.disambiguator = name
    print(f'{name:5s}  programming -> {entity(kbel.link(mention)[0]):22s}  snake -> {entity(kbel.link(snake)[0])}')

naive  programming -> Python (Q28865)         snake -> Python (Q28865)
sim    programming -> Python (Q28865)         snake -> python (Q184018)


`sim` also returns a ranked list with scores; `limit` sets how many results:

In [35]:
for result in kbel.link(snake, limit=3):
    print(f'{result.score:.3f}  {entity(result):30s} {result.candidate.description}')

0.684  python (Q184018)               family of snakes
0.612  Burmese python (Q2716137)      species of large nonvenomous snake
0.610  Python (Q271218)               genus of reptiles


### Linking properties

With `entity_type=EntityType.PROPERTY` the search looks for properties
(relations) instead of items:

In [36]:
for term in ('place of birth', 'born in', 'created by'):
    [result] = kbel.link(Mention(term=term, text='', entity_type=EntityType.PROPERTY), limit=1)
    print(f'{term:15s} -> {entity(result)}')

place of birth  -> place of birth (P19)
born in         -> place of birth (P19)
created by      -> creator (P170)


The knowledge base is also chosen by name — e.g. `wikidata-vdb`, a semantic
(vector) search over Wikidata:

In [37]:
kbel.knowledge_base = 'wikidata-vdb'
print(entity(kbel.link(mention, limit=1)[0]))

Python (Q28865)


## 2. `extract`: from text to triples

An **extractor** turns text into triples. The two used here work differently:

- `mrebel` is a neural model trained on Wikidata: predicates are Wikidata
  property labels (`place of birth`), and it supports 18 languages.
- `openie` is Stanford OpenIE (rule based, requires Java): predicates are
  phrases from the text (`was born in`).

In [38]:
TEXT = ('Marie Curie was a Polish and naturalised-French physicist and chemist who was born in Warsaw. '
        'Marie Curie won the Nobel Prize in Physics in 1903.')

kbel = KBEL('wikidata', extractor='mrebel')

for triple in kbel.extract(TEXT):
    print(triple)

Triple(subject='Marie Curie', predicate='place of birth', object='Warsaw')
Triple(subject='Marie Curie', predicate='occupation', object='physicist')
Triple(subject='Marie Curie', predicate='occupation', object='chemist')
Triple(subject='Marie Curie', predicate='country of citizenship', object='French')
Triple(subject='Marie Curie', predicate='award received', object='Nobel Prize in Physics')
Triple(subject='Nobel Prize in Physics', predicate='winner', object='Marie Curie')
Triple(subject='Marie Curie', predicate='floruit', object='1903')


In [39]:
kbel.extractor = 'openie'

for triple in kbel.extract(TEXT):
    print(triple)

Triple(subject='Marie Curie', predicate='was', object='Polish physicist')
Triple(subject='chemist', predicate='was born in', object='Warsaw')
Triple(subject='Marie Curie', predicate='was physicist', object='born in Warsaw')
Triple(subject='Marie Curie', predicate='was Polish physicist', object='born in Warsaw')
Triple(subject='Marie Curie', predicate='was Polish and naturalised-French physicist', object='born in Warsaw')
Triple(subject='Curie', predicate='was Polish', object='born')
Triple(subject='chemist', predicate='was', object='born')
Triple(subject='Marie Curie', predicate='won Nobel Prize in', object='1903')
Triple(subject='Marie Curie', predicate='won in', object='1903')
Triple(subject='Marie Curie', predicate='won Nobel Prize in', object='Physics')
Triple(subject='Marie Curie', predicate='won', object='Nobel Prize')
Triple(subject='Marie Curie', predicate='won in', object='Physics')


Extractors take their own options. For instance, `mrebel` can read other
languages — the predicates still come back as English Wikidata labels:

In [40]:
from kbel.extractors import TripleExtractor

kbel.extractor = TripleExtractor('mrebel', source_language='pt_XX')

for triple in kbel.extract('Ada Lovelace trabalhou com Charles Babbage na Máquina Analítica, em Londres.'):
    print(triple)

Triple(subject='Máquina Analítica', predicate='discoverer or inventor', object='Charles Babbage')
Triple(subject='Charles Babbage', predicate='notable work', object='Máquina Analítica')


## 3. `construct`: from a document to KIF statements

`construct` combines both: it splits the document into paragraphs and
sentences, extracts triples from each sentence, links subject (item),
predicate (property) and object (item) — using the sentence as the mention's
`text` and the paragraph as its `context` — and yields a
[KIF](https://github.com/PUC-Behring-AI/kif) **`Statement`** for every triple
whose three parts were linked:

```
Statement(subject, ValueSnak(property, value))
```

In [6]:
%env KIF_DEBUG=
%env KIF_RESOLVE_ENTITIES=1
from kif_lib import *
from kif_lib.vocabulary import wd

DOCUMENT = """The Austroasiatic languages, in recent classifications synonymous with Mon–Khmer, are a large language family of continental Southeast Asia, also scattered throughout India, Bangladesh, Nepal and the southern border of China. The name Austroasiatic comes from the Latin words for "south" and "Asia", hence "South Asia". Of these languages, only Vietnamese, Khmer, and Mon have a long-established recorded history, and only Vietnamese and Khmer have official status (in Vietnam and Cambodia, respectively). The rest of the languages are spoken by minority groups. Ethnologue identifies 168 Austroasiatic languages. These form thirteen established families (plus perhaps Shompen, which is poorly attested, as a fourteenth), which have traditionally been grouped into two, as Mon–Khmer and Munda. However, one recent classification posits three groups (Munda, Nuclear Mon-Khmer and Khasi-Khmuic) while another has abandoned Mon–Khmer as a taxon altogether, making it synonymous with the larger family. Austroasiatic languages have a disjunct distribution across India, Bangladesh, Nepal and Southeast Asia, separated by regions where other languages are spoken. They appear to be the autochthonous languages of Southeast Asia, with the neighboring Indo-Aryan, Tai–Kadai, Dravidian, Austronesian, and Sino-Tibetan languages being the result of later migrations.
"""

kbel = KBEL('wikidata', disambiguator='naive', extractor='mrebel')

statements = kbel.construct(DOCUMENT, language='en')

display(*statements)

env: KIF_DEBUG=
env: KIF_RESOLVE_ENTITIES=1


(**Statement** (**Item** [wd:Q33199](http://www.wikidata.org/entity/Q33199)) (**ValueSnak** (**Property** [instance of](http://www.wikidata.org/entity/P31)) (**Item** [wd:Q25295](http://www.wikidata.org/entity/Q25295))))

(**Statement** (**Item** [wd:Q8573286](http://www.wikidata.org/entity/Q8573286)) (**ValueSnak** (**Property** [instance of](http://www.wikidata.org/entity/P31)) (**Item** [wd:Q25295](http://www.wikidata.org/entity/Q25295))))

(**Statement** (**Item** [wd:Q15753367](http://www.wikidata.org/entity/Q15753367)) (**ValueSnak** (**Property** [part of](http://www.wikidata.org/entity/P361)) (**Item** [Asia](http://www.wikidata.org/entity/Q48))))

(**Statement** (**Item** [wd:Q33199](http://www.wikidata.org/entity/Q33199)) (**ValueSnak** (**Property** [located in/on physical feature](http://www.wikidata.org/entity/P706)) (**Item** [wd:Q15753367](http://www.wikidata.org/entity/Q15753367))))

(**Statement** (**Item** [wd:Q33199](http://www.wikidata.org/entity/Q33199)) (**ValueSnak** (**Property** [said to be the same as](http://www.wikidata.org/entity/P460)) (**Item** [wd:Q15753367](http://www.wikidata.org/entity/Q15753367))))

... (11 outputs left out)

(**Statement** (**Item** [wd:Q8573286](http://www.wikidata.org/entity/Q8573286)) (**ValueSnak** (**Property** [instance of](http://www.wikidata.org/entity/P31)) (**Item** [taxon](http://www.wikidata.org/entity/Q16521))))

(**Statement** (**Item** [wd:Q33199](http://www.wikidata.org/entity/Q33199)) (**ValueSnak** (**Property** [subclass of](http://www.wikidata.org/entity/P279)) (**Item** [wd:Q50816272](http://www.wikidata.org/entity/Q50816272))))

(**Statement** (**Item** [Nepal](http://www.wikidata.org/entity/Q837)) (**ValueSnak** (**Property** [located in/on physical feature](http://www.wikidata.org/entity/P706)) (**Item** [wd:Q11708](http://www.wikidata.org/entity/Q11708))))

(**Statement** (**Item** [Bangladesh](http://www.wikidata.org/entity/Q902)) (**ValueSnak** (**Property** [part of](http://www.wikidata.org/entity/P361)) (**Item** [wd:Q11708](http://www.wikidata.org/entity/Q11708))))

In [2]:
from langchain_openrouter import ChatOpenRouter
model = ChatOpenRouter(model='openai/gpt-4.1', api_key=os.environ['LLM_API_KEY'])

In [ ]:
from kbel.disambiguators import Disambiguator
from kbel.extractors import TripleExtractor
from kbel.disambiguators.llm import LLM_Disambiguator
from kbel.extractors.llm import LLMTripleExtractor

kbel = KBEL('wikidata-vdb')
kbel.disambiguator = Disambiguator('llm', model= model)
kbel.extractor = TripleExtractor('llm', model=model)

In [ ]:


DOCUMENT = """The Austroasiatic languages, in recent classifications synonymous with Mon–Khmer, are a large language family of continental Southeast Asia, also scattered throughout India, Bangladesh, Nepal and the southern border of China. The name Austroasiatic comes from the Latin words for "south" and "Asia", hence "South Asia". Of these languages, only Vietnamese, Khmer, and Mon have a long-established recorded history, and only Vietnamese and Khmer have official status (in Vietnam and Cambodia, respectively). The rest of the languages are spoken by minority groups. Ethnologue identifies 168 Austroasiatic languages. These form thirteen established families (plus perhaps Shompen, which is poorly attested, as a fourteenth), which have traditionally been grouped into two, as Mon–Khmer and Munda. However, one recent classification posits three groups (Munda, Nuclear Mon-Khmer and Khasi-Khmuic) while another has abandoned Mon–Khmer as a taxon altogether, making it synonymous with the larger family. Austroasiatic languages have a disjunct distribution across India, Bangladesh, Nepal and Southeast Asia, separated by regions where other languages are spoken. They appear to be the autochthonous languages of Southeast Asia, with the neighboring Indo-Aryan, Tai–Kadai, Dravidian, Austronesian, and Sino-Tibetan languages being the result of later migrations.
"""

statements = kbel.construct(DOCUMENT, language='en')

display(*statements)


env: KIF_DEBUG=
env: KIF_RESOLVE_ENTITIES=1


(**Statement** (**Item** https://www.wikidata.org/entity/Q33199) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P4970) (**Item** https://www.wikidata.org/entity/Q337657)))

(**Statement** (**Item** https://www.wikidata.org/entity/Q33199) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P279) (**Item** https://www.wikidata.org/entity/Q25295)))

(**Statement** (**Item** https://www.wikidata.org/entity/Q33199) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P131) (**Item** https://www.wikidata.org/entity/Q43467)))

(**Statement** (**Item** https://www.wikidata.org/entity/Q33199) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P131) (**Item** https://www.wikidata.org/entity/Q668)))

(**Statement** (**Item** https://www.wikidata.org/entity/Q33199) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P131) (**Item** https://www.wikidata.org/entity/Q902)))

... (32 outputs left out)

(**Statement** (**Item** https://www.wikidata.org/entity/Q49228) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P31) (**Item** https://www.wikidata.org/entity/Q34770)))

(**Statement** (**Item** https://www.wikidata.org/entity/Q49228) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P1536) (**Item** https://www.wikidata.org/entity/Q177626)))

(**Statement** (**Item** https://www.wikidata.org/entity/Q45961) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P31) (**Item** https://www.wikidata.org/entity/Q34770)))

(**Statement** (**Item** https://www.wikidata.org/entity/Q45961) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P1536) (**Item** https://www.wikidata.org/entity/Q177626)))

### Async

Extraction, searches and disambiguations run concurrently (at most
`max_concurrency` at a time). In async code — Jupyter included — iterate
the async version:

In [6]:
async for statement in kbel.aconstruct(DOCUMENT, language='en', max_concurrency=8):
    display(statement)

(**Statement** (**Item** https://www.wikidata.org/entity/Q33199) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P4970) (**Item** https://www.wikidata.org/entity/Q337657)))

(**Statement** (**Item** https://www.wikidata.org/entity/Q33199) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P31) (**Item** https://www.wikidata.org/entity/Q25295)))

(**Statement** (**Item** https://www.wikidata.org/entity/Q33199) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P131) (**Item** https://www.wikidata.org/entity/Q43467)))

(**Statement** (**Item** https://www.wikidata.org/entity/Q33199) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P131) (**Item** https://www.wikidata.org/entity/Q668)))

(**Statement** (**Item** https://www.wikidata.org/entity/Q33199) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P131) (**Item** https://www.wikidata.org/entity/Q902)))

(**Statement** (**Item** https://www.wikidata.org/entity/Q33199) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P131) (**Item** https://www.wikidata.org/entity/Q837)))

... (24 outputs left out)

(**Statement** (**Item** https://www.wikidata.org/entity/Q34171) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P828) (**Item** https://www.wikidata.org/entity/Q177626)))

(**Statement** (**Item** https://www.wikidata.org/entity/Q33311) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P828) (**Item** https://www.wikidata.org/entity/Q177626)))

(**Statement** (**Item** https://www.wikidata.org/entity/Q49228) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P828) (**Item** https://www.wikidata.org/entity/Q177626)))

(**Statement** (**Item** https://www.wikidata.org/entity/Q45961) (**ValueSnak** (**Property** https://www.wikidata.org/entity/P828) (**Item** https://www.wikidata.org/entity/Q177626)))

`link` has an async version too, so several mentions can be linked at once:

In [ ]:
import asyncio

mentions = [Mention(term=t, text='') for t in ('Marie Curie', 'Warsaw', 'polonium', 'radium')]
results = await asyncio.gather(*(kbel.alink(m, limit=1) for m in mentions))

for m, [result] in zip(mentions, results):
    print(f'{m.term:12s} -> {entity(result)}')

Marie Curie  -> Marie Curie (Q7186)
Warsaw       -> Warsaw (Q270)
polonium     -> polonium (Q979)
radium       -> radium (Q1128)


Any combination of knowledge base, extractor and disambiguator works with
these methods — for example `KBEL('wikidata-vdb', disambiguator='sim',
extractor='openie')`.